***DEMO***

This example demonstrates redaction, advanced analysis, data normalization, and hashing

This demo using a short call transcript: 


Agent: Thanks for calling TechNova Support, voted New Yorks best Technology provider in 2025. How can I help you today?
Caller: Hi this is Bob Smith calling you can call me Robert. I uhh bought the TechNova EchoStream Pro back in uhh 2023, I need to buy a new charger for my daughter who is 12 because she was using the old 2018 model.
Agent: No problem mister Smith. Can I get your date of birth to verify your account? and it looks like you have 3 devices on your account can you give me the serial number?
Caller: Im turning 40 this year. My birth day is oct 29 1985. Oh and the serial number is nine-eight-seven6-five4-three two-one-zero
Agent: Got it. I'm pulling up your account and i can see that card used for the purchase was a TechNova Platinum Advantage card can you give me the card number and ill place the order for you.
Caller: Sure its .. one second.. the card number is 4591 3334 2334 5533
Agent: Thank you just to confirm this matches the card we had on file ending in 5533. I will place the order for you and it will show up on your next statement for account number 8889991234



We have references to years, product names, multiple names used for the same person, ages, date of birth and some PCI info and potentially sensitive account info. 
In the text you can see disfluencies and some other small little hiccups reflecting the way weird and wonderful ways that people talk. 
We do partial redaction on product names and credit cards, we do anonymization while maintaining statistical utility for years, ages, and date of birth. 
Years are now referenced by decade
Ages are now age ranges
Date of birth is now season and year range (northern hemisphere but this is just to illustrate that you are not limited with options for how this is can be done)
Credit cards have partial redaction
Account numbers have a one way hash appended



In [3]:
# This code assumes that you have the Private AI deidentification service running locally on port 8080 
# or you have a valid community api key saved in a .env file in the project folder that contains the variable
# PRIVATEAI_API_KEY = my_key 
# It also assumes that you have installed the Private AI python client.

#region SETUP - Imports for our Private AI python SDK helper libraries
from privateai_client import PAIClient
from privateai_client.post_processing import deidentify_text
from privateai_client.post_processing.processors import MarkerEntityProcessor
from privateai_client.components import AnalyzeTextRequest, ProcessTextRequest
import os
import dotenv
import json
import math
import hashlib
from datetime import datetime

#endregion

#region SETUP - Initialize test data
text = ['''Agent: Thanks for calling TechNova Support, voted New Yorks best Technology provider in 2025. How can I help you today?
Caller: Hi this is Bob Smith calling you can call me Robert. I uhh bought the TechNova EchoStream Pro back in uhh 2023, I need to buy a new charger for my daughter who is 12 because she was using the old 2018 model.
Agent: No problem mister Smith. Can I get your date of birth to verify your account? and it looks like you have 3 devices on your account can you give me the serial number?
Caller: Im turning 40 this year. My birth day is oct 29 1985. Oh and the serial number is nine-eight-seven6-five4-three two-one-zero
Agent: Got it. I'm pulling up your account and i can see that card used for the purchase was a TechNova Platinum Advantage card can you give me the card number and ill place the order for you.
Caller: Sure its .. one second.. the card number is 4591 3334 2334 5533
Agent: Thank you just to confirm this matches the card we had on file ending in 5533. I will place the order for you now for your account number 8889991 234
''']

#endregion

###----------------------------------------------------------------------------###
#region Prepare and issue Private AI call
###----------------------------------------------------------------------------###
### PRIVATE AI API CALL
dotenv.load_dotenv()
client = PAIClient("https", 'api.private-ai.com/community/v4' ) 
#client = PAIClient("http", 'localhost:8000' )
client.add_api_key(os.environ["PRIVATEAI_API_KEY"])

redact_request = {
    "text": text,
    "entity_detection": {
        "return_entity": True,
        "entity_types": [
        {
            "type": "ENABLE",
            "value": [#"ACCOUNT_NUMBER",
                    "DRIVER_LICENSE","DURATION","EMAIL_ADDRESS","EVENT","FILENAME","GENDER","HEALTHCARE_NUMBER",
                    #"DOB",
                    "IP_ADDRESS","LANGUAGE","LOCATION","LOCATION_ADDRESS","LOCATION_ADDRESS_STREET","LOCATION_CITY","LOCATION_COORDINATE","LOCATION_COUNTRY","LOCATION_STATE",
                    "LOCATION_ZIP","MARITAL_STATUS","MONEY","NAME","NAME_FAMILY","NAME_GIVEN","NAME_MEDICAL_PROFESSIONAL","NUMERICAL_PII","OCCUPATION","ORGANIZATION",
                    "ORGANIZATION_MEDICAL_FACILITY","ORIGIN","PASSPORT_NUMBER","PASSWORD","PHONE_NUMBER","PHYSICAL_ATTRIBUTE","POLITICAL_AFFILIATION","RELIGION",
                    "SEXUALITY","SSN","TIME","URL","USERNAME","VEHICLE_ID","ZODIAC_SIGN","BLOOD_TYPE","CONDITION","DOSE","DRUG","INJURY","MEDICAL_PROCESS",
                    "STATISTICS","BANK_ACCOUNT",
                    #"CREDIT_CARD","CREDIT_CARD_EXPIRATION","CVV",
                    "ROUTING_NUMBER",
                    
                    ###Latest BETA entities####
                    "CORPORATE_ACTION","EFFECT", "FINANCIAL_METRIC","MEDICAL_CODE","ORGANIZATION_ID",
                    "PROJECT","TREND"]
        }
        ]
    }
}
#Redaction call
text_request = ProcessTextRequest.fromdict(redact_request)
redact_resp = client.process_text(text_request)
redacted_text = redact_resp.processed_text

#Analyze call
request = {"text": redacted_text, "entity_detection": {"entity_types": [{"type": "ENABLE", "value": ["AGE", "YEAR", "DATE", "PRODUCT", "DOB","CREDIT_CARD","ACCOUNT_NUMBER"]}]}}
analyze_request = AnalyzeTextRequest.fromdict(request)
resp = client.analyze_text(analyze_request)


### PRIVATE AI API CALL SECTION END
###----------------------------------------------------------------------------###
#endregion
###----------------------------------------------------------------------------###

###----------------------------------------------------------------------------###
#region BUSINESS RULES - Customizable redaction logic
def redact_by_rounding(entity) -> str:
    """Round to the closest 10th"""
    if "formatted" in entity["analysis_result"]:
        age = int(entity["analysis_result"]["formatted"])
        
        rounded_age = int(round(age * 10, -2) / 10)
        if (rounded_age == 0):
            rounded_age = 1

        if rounded_age < age:
            return f"\033[92m{str(rounded_age)}-{str(rounded_age + 10)}\033[0m"
        else:
            return f"\033[92m{str(rounded_age - 10)}-{str(rounded_age)}\033[0m"
    else:
        return "#"
    

def redact_year_by_rounding(entity) -> str:
    """Round to the closest 10th"""
    if "formatted" in entity["analysis_result"]:
        year = int(entity["analysis_result"]["formatted"])        
        rounded_year = int(math.floor(year / 10) * 10)
        return f"\033[91m{str(rounded_year)}'s\033[0m"
    else:
        return "#"
    
def redact_products(entity) -> str:    
    if entity["text"]:
        product = entity["text"]
        return f"\033[94m{' '.join(word[0] + 'X' * (len(word) - 1) if word else '' for word in product.split())}\033[0m"        
    else:
        return "#"
    

def date_to_season_label(entity) -> str:

    if "subtypes" in entity["analysis_result"]:       

        month = 0
        year = 0

        for item in entity["analysis_result"]["subtypes"]:
            if item["label"] == "MONTH":
                month = item["formatted"] 
            if item["label"] == "YEAR":
                year = int(item["formatted"])
         
        # Define season boundaries (Northern Hemisphere)
        if month in [12, 1, 2]:
            season = "WINTER"
            if month == 12:
                year += 1  # December belongs to the winter of the next year
        elif month in [3, 4, 5]:
            season = "SPRING"
        elif month in [6, 7, 8]:
            season = "SUMMER"
        else:
            season = "FALL"

        rounded_year = int(5 * round(year / 5))

        rounded_year_text = ''
        if (rounded_year == 0):
            rounded_year = 1

        if rounded_year < year:
            rounded_year_text = f"{str(rounded_year)}-{str(rounded_year + 5)}"
        else:
            rounded_year_text = f"{str(rounded_year - 5)}-{str(rounded_year)}"
        
        return f"\033[35m{season} {rounded_year_text}\033[0m"
    else: 
        return "#"

def last_4_cc(entity) -> str: 
    result = "[CREDIT_CARD]"
    if "formatted" in entity["analysis_result"]:
        cc =  entity["analysis_result"]["formatted"]
        parts = cc.split()
        if len(parts) == 4:
            parts[:3] = ['****', '****', '****']
            formatted_cc = ' '.join(parts)
            result = f"\033[96m{formatted_cc}\033[0m"        
    elif "text" in entity:
        cc =  entity["text"]
        parts = cc.split()
        if len(parts) == 1:
            formatted_cc = ''.join(parts)
            result = f"\033[96m{formatted_cc}\033[0m"
    
    return result

def acct_num_hash(entity)-> str:
    result = "[ACCOUNT_NUMBER]"
    if "text" in entity:
        acct_num =  entity["text"]
        acct_num = "".join(acct_num.split())
        salt = 'my_salt_value'
        hash_object = hashlib.sha256((salt + acct_num).encode('utf-8'))
        hashed = hash_object.hexdigest()

        result = f"\033[96m[ACCOUNT_NUMBER_{hashed}]\033[0m"        
    
    return result
    

    
#endregion Customizable redaction logic
###----------------------------------------------------------------------------###

###----------------------------------------------------------------------------###
#region APPLY BUSINESS RULES - Private AI helper code
###----------------------------------------------------------------------------###
### PRIVATE AI HELPER CODE
#Build redacted result
entity_processor = {"AGE": redact_by_rounding, "YEAR": redact_year_by_rounding, "PRODUCT": redact_products, "DOB": date_to_season_label, "CREDIT_CARD": last_4_cc, "ACCOUNT_NUMBER": acct_num_hash}
deidentified_text = []
if resp is not None:
    # Ensure redacted_text is a list of strings
    if isinstance(redacted_text, list) and all(isinstance(item, str) for item in redacted_text):
        deidentified_text = deidentify_text(redacted_text, resp, entity_processors=entity_processor, default_processor=MarkerEntityProcessor())
    else:
        raise ValueError("redacted_text must be a list of strings")
### PRIVATE AI HELPER CODE
###----------------------------------------------------------------------------###
#endregion
###----------------------------------------------------------------------------###

#region print interwoven results
for original_line, processed_line in zip(text[0].splitlines(), deidentified_text[0].splitlines()):
    if not original_line.strip() and not processed_line.strip():
        continue
    print(f"ORIGINAL:  {original_line}\nPROCESSED: {processed_line}\n")
#endregion    

ORIGINAL:  Agent: Thanks for calling TechNova Support, voted New Yorks best Technology provider in 2025. How can I help you today?
PROCESSED: [OCCUPATION_1]: Thanks for calling [ORGANIZATION_1], voted [LOCATION_STATE_1] best Technology provider in 2020's. How can I help you today?

ORIGINAL:  Caller: Hi this is Bob Smith calling you can call me Robert. I uhh bought the TechNova EchoStream Pro back in uhh 2023, I need to buy a new charger for my daughter who is 12 because she was using the old 2018 model.
PROCESSED: Caller: Hi this is [NAME_1] calling you can call me [NAME_GIVEN_1]. I uhh bought the [ORGANIZATION_2] EXXXXXXXXX PXX back in uhh 2020's, I need to buy a new charger for my daughter who is 10-20 because she was using the old 2010's model.

ORIGINAL:  Agent: No problem mister Smith. Can I get your date of birth to verify your account? and it looks like you have 3 devices on your account can you give me the serial number?
PROCESSED: [OCCUPATION_1]: No problem mister [NAME_FAMIL